In [ ]:
from pathlib import Path
import sys

# Config path (Colab vs VS Code locale)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/HPC/finale')
else:
    # In locale: metti qui la cartella base dove hai modello/dataset (default = cartella del workspace)
    ROOT = Path.cwd()

print('IN_COLAB =', IN_COLAB)
print('ROOT =', ROOT)

In [ ]:
# In locale evita di reinstallare TensorFlow ad ogni run.
# Se ti serve forzare TF 2.15, installa UNA volta nel tuo venv: `pip install tensorflow==2.15.0`

try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
except Exception as e:
    raise RuntimeError(
        'TensorFlow non trovato in questo kernel. Installa nel tuo ambiente e riavvia il kernel.'
    ) from e

# Info utili per capire la lentezza (CPU vs GPU)
print('GPUs:', tf.config.list_physical_devices('GPU'))

# ***CARICA I MODELLI***

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, GlobalAveragePooling2D, Dropout, Dense, SeparableConv2D, BatchNormalization
from tensorflow.keras.initializers import Constant
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model

class_names = ['ANGRY', 'DISGUST', 'FEAR', 'HAPPY', 'NEUTRAL', 'SAD', 'SURPRISE']

class ExpandDimsLayer(Layer):
    def __init__(self, axis, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.expand_dims(inputs, axis=self.axis)

class SqueezeLayer(Layer):
    def __init__(self, axis, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.squeeze(inputs, axis=self.axis)

# (Opzionale) builder: il notebook lavora principalmente caricando il pretrained ConvNeXt
def build_convnext_model(learning_rate, dropout_rate, l2_reg, initial_bias):
    num_classes = 7
    img_shape = (128, 128, 3)
    input_layer = tf.keras.Input(shape=img_shape, name='universal_input')

    backbone = tf.keras.applications.ConvNeXtBase(
        include_top=False,
        include_preprocessing=True,
        weights='imagenet',
        input_shape=img_shape,
    )
    base_model = Model(
        backbone.input,
        backbone.get_layer('convnext_base_stage_2_block_24_identity').output,
        name='base_model',
    )
    base_model.trainable = False

    self_attention = tf.keras.layers.Attention(use_scale=True, name='attention')
    patch_extraction = tf.keras.Sequential([
        SeparableConv2D(256, kernel_size=4, strides=4, padding='same', activation='relu'),
        SeparableConv2D(256, kernel_size=2, strides=2, padding='valid', activation='relu'),
        tf.keras.layers.Conv2D(256, kernel_size=1, strides=1, padding='valid', activation='relu', kernel_regularizer=l2(l2_reg)),
    ], name='patch_extraction')

    global_average_layer = GlobalAveragePooling2D(name='gap')
    pre_classification = tf.keras.Sequential([
        Dense(32, activation='relu', kernel_regularizer=l2(l2_reg)),
        BatchNormalization(),
    ], name='pre_classification')
    prediction_layer = Dense(num_classes, activation='softmax', name='classification_head', bias_initializer=Constant(initial_bias))

    x = base_model(input_layer, training=False)
    x = patch_extraction(x)
    x = global_average_layer(x)
    x = Dropout(dropout_rate)(x)
    x = pre_classification(x)
    x = ExpandDimsLayer(axis=-1)(x)
    x = self_attention([x, x])
    x = SqueezeLayer(axis=-1)(x)
    outputs = prediction_layer(x)

    model = Model(inputs=input_layer, outputs=outputs, name='convnext-train-head')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, global_clipnorm=3.0),
        loss=categorical_focal_loss(alpha=0.25, gamma=2.0),
        metrics=['categorical_accuracy'],
    )
    return model


In [ ]:
import numpy as np
import tensorflow as tf

def categorical_focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        focal_loss = weight * cross_entropy
        return tf.reduce_sum(focal_loss, axis=-1)
    return loss

# Caricamento SOLO del modello ConvNeXt
convnext_path = '/content/drive/MyDrive/Colab Notebooks/HPC/finale/model/pretrained_ConvNeXt_finetuning'

custom_objects = {
    'loss': categorical_focal_loss(alpha=0.25, gamma=2.0),
    'categorical_focal_loss': categorical_focal_loss,
    'ExpandDimsLayer': ExpandDimsLayer,
    'SqueezeLayer': SqueezeLayer,
}

with tf.keras.utils.custom_object_scope(custom_objects):
    convnext = tf.keras.models.load_model(convnext_path)

convnext.summary(show_trainable=True)


In [ ]:
####### per abilitare i generatori di immagini


import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from sklearn.utils import shuffle

class CustomBalancedDataGenerator(Sequence):
    def __init__(self, x_data, y_data, batch_size, augmentations=None, data_inf=None, label_smoothing=0.1,paths_data=None, **kwargs):
        super().__init__(**kwargs)
        self.x_data = x_data
        self.y_data = y_data
        self.batch_size = batch_size
        self.data_inf = data_inf
        self.label_smoothing = label_smoothing
        self.indices = np.arange(len(x_data))
        self.paths_data = paths_data


        # Se siamo in 'train' o 'valid', impostiamo le augmentation e il bilanciamento
        if data_inf in ['train', 'valid']:
            #print(y_data)
            self.augmentations = ImageDataGenerator(**augmentations)
            self.classes = np.unique(np.argmax(y_data, axis=1))  # Ricaviamo le classi dai dati one-hot encoded
            self.class_indices = {cls: np.where(np.argmax(y_data, axis=1) == cls)[0] for cls in self.classes}
            self.num_classes = len(self.classes)
            self.samples_per_class = max(1, self.batch_size // self.num_classes)

            # Coda ciclica per le classi minoritarie
            self.class_pointers = {cls: 0 for cls in self.classes}

        # Se siamo in 'test', usiamo solo rescale e nessuna augmentation o bilanciamento
        elif data_inf == 'test':
            self.augmentations = ImageDataGenerator(**(augmentations or {}))
        self.index = 0
        self.on_epoch_end()
        print(f"Generator initialized: {data_inf} mode")

    def __len__(self):
        return int(np.ceil(len(self.x_data) / self.batch_size))
    def __next__(self):
        # Il comportamento dell'iteratore
        if self.index >= len(self):
            raise StopIteration
        batch = self.__getitem__(self.index)
        self.index += 1
        return batch

    def __iter__(self):
        # Rende l'oggetto un iteratore
        self.index = 0
        return self
    def __getitem__(self, index):
        if self.data_inf == 'test':
            # Per il test set, usiamo semplicemente gli indici
            start_idx = index * self.batch_size
            end_idx = min((index + 1) * self.batch_size, len(self.x_data))
            batch_x = self.x_data[start_idx:end_idx]
            batch_y = self.y_data[start_idx:end_idx]
            batch_x = np.array(batch_x)
            batch_y = np.array(batch_y)
            # Se hai i path
            if self.paths_data is not None:
                batch_paths = self.paths_data[start_idx:end_idx]
            else:
                batch_paths = None
        else:
            # Per train/valid, selezioniamo batch bilanciati
            batch_x, batch_y = [], []
            for cls in self.classes:
                cls_indices = self.class_indices[cls]
                cls_pointer = self.class_pointers[cls]

                # Seleziona i dati dalla coda ciclica
                selected_indices = cls_indices[cls_pointer:cls_pointer + self.samples_per_class]
                batch_x.extend(self.x_data[selected_indices])
                batch_y.extend(self.y_data[selected_indices])

                # Aggiorna il puntatore per la classe
                self.class_pointers[cls] += len(selected_indices)

                # Se abbiamo esaurito i dati per la classe, fai uno shuffle e riparti
                if self.class_pointers[cls] >= len(cls_indices):
                    self.class_pointers[cls] = 0
                    np.random.shuffle(cls_indices)  # Shuffle della classe
                    self.class_indices[cls] = cls_indices

            batch_x = np.array(batch_x)
            batch_y = np.array(batch_y)
            batch_x, batch_y = shuffle(batch_x, batch_y)

            # Applica il label smoothing
            if self.label_smoothing > 0:
                batch_y = self.apply_label_smoothing(batch_y)
            batch_paths = None



        # Applica il rescale o le trasformazioni per augmentation
        augmented_batch_x = np.zeros_like(batch_x)
        for i in range(len(batch_x)):
            augmented_batch_x[i] = self.augmentations.random_transform(batch_x[i])

        return augmented_batch_x, batch_y


    def on_epoch_end(self):
        if self.data_inf != 'test':
            print("Epoch ended. Shuffling data.")
            for cls in self.classes:
                np.random.shuffle(self.class_indices[cls])  # Shuffle degli indici per ogni classe

    def apply_label_smoothing(self, labels):
        """Applica il label smoothing alle etichette one-hot"""
        if self.label_smoothing > 0:
            labels = labels.astype(np.float32)  # Assicurati che sia in formato float
            num_classes = labels.shape[1]  # Ottieni il numero di classi (assumendo one-hot encoding)
            smooth_value = self.label_smoothing / (num_classes - 1)  # Calcolo del valore per le classi non corrette
            smoothed_labels = np.ones_like(labels, dtype=np.float32) * smooth_value  # Etichette smussate per tutte le classi
            for i in range(len(labels)):
                true_class = np.argmax(labels[i])  # Ottieni la classe corretta (indice della classe 1)
                smoothed_labels[i, true_class] = 1.0 - self.label_smoothing  # Imposta la probabilità della classe corretta
            return smoothed_labels
        else:
            return labels


import os
import numpy as np
import h5py
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical
import cv2

def load_data_and_labels(file_path, info):
    class_names = None
    with h5py.File(file_path, 'r') as f:
        if info == 'train':
            X_train = np.array(f['X_train'])
            y_train = np.array(f['y_train'])
            X_val = np.array(f['X_val'])
            y_val = np.array(f['y_val'])
            class_names = [name.decode('utf-8') for name in f['class_names']]
            return X_train, y_train, X_val, y_val, class_names
        else:
            # CASO TEST
            x = np.array(f['X_test'])
            y = np.array(f['y_test'])
            # Leggiamo anche i path se esistono
            if 'paths' in f:
                # Se 'paths' è un dataset di stringhe a lunghezza variabile
                # con h5py.string_dtype, possiamo leggerlo direttamente:
                paths_data = f['paths'][...]  # np array di stringhe
            else:
                paths_data = None
            return x, y, class_names, paths_data

# Funzione per creare i generatori di dati
def carica_dati():
    file_path = '/content/drive/MyDrive/Colab Notebooks/HPC/finale/dataset' #### il path dove metti i .h5 che ti invio
    test_path = '/content/drive/MyDrive/Colab Notebooks/datasets/dataset_giusto/test_path.h5' # per il test
    path = os.path.join(file_path, 'dataset.h5') #per train e val

    X_test, y_test,  class_names, test_paths = load_data_and_labels(test_path, 'test')
    X_train, y_train, X_val, y_val, class_names = load_data_and_labels(path, 'train')

    augmentations = {
        'rotation_range': 10,
        'width_shift_range': 0.2,
        'shear_range': 0.3,
        'horizontal_flip': True,
        'fill_mode': 'wrap',
    }
    test_augmentations = {}
    NUM_CLASSES = 7
# Conversione delle etichette in one-hot encoding
    y_train_one_hot = to_categorical(y_train, num_classes=NUM_CLASSES)
    y_val_one_hot = to_categorical(y_val, num_classes=NUM_CLASSES)
    y_test_one_hot = to_categorical(y_test, num_classes=NUM_CLASSES)
    train_generator_focal_smoot = CustomBalancedDataGenerator(
        X_train,
        y_train_one_hot,
        batch_size=64,
        augmentations=augmentations,
        data_inf='train',
        label_smoothing=0.05,
        paths_data=None  # se non hai i path
    )

    valid_generator_focal_smoot = CustomBalancedDataGenerator(
        X_val,
        y_val_one_hot,
        batch_size=64,
        augmentations=augmentations,
        data_inf='valid',
        label_smoothing=0,
        paths_data=None
    )
    # Passi test_paths al generator di test
    test_generator_focal_smoot = CustomBalancedDataGenerator(
        x_data=X_test,
        y_data=y_test_one_hot,
        batch_size=64,
        augmentations=test_augmentations,
        data_inf='test',
        label_smoothing=0,
        paths_data=test_paths  # Nuovo parametro
    )

    return test_generator_focal_smoot,train_generator_focal_smoot,valid_generator_focal_smoot

test_generator_focal_smoot, train_generator_focal_smoot,valid_generator_focal_smoot = carica_dati()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def evaluate_keras_model(model, test_generator, model_name):
    class_names_fixed = ['ANGER', 'DISGUST', 'FEAR', 'HAPPINESS', 'NEUTRALITY', 'SADNESS', 'SURPRISE']

    y_true = np.argmax(test_generator.y_data, axis=1)
    test_generator.in_evaluate_mode = True
    probabilities = model.predict(test_generator, verbose=1)
    test_generator.in_evaluate_mode = False

    y_pred = np.argmax(probabilities, axis=1)
    confidences = np.max(probabilities, axis=1)

    threshold = 0.6
    high_conf_wrong = (y_pred != y_true) & (confidences >= threshold)

    print("Errori con alta confidenza:")
    error_indices = np.where(high_conf_wrong)[0]
    for idx in error_indices:
        print(f"Immagine {idx}: Pred: {class_names_fixed[y_pred[idx]]} "
              f"(Conf: {confidences[idx]:.2f}) - Reale: {class_names_fixed[y_true[idx]]}")

    sorted_indices = error_indices[np.argsort(-confidences[error_indices])]
    num_errors = len(sorted_indices)
    print(f"Numero totale di errori ad alta confidenza: {num_errors}")

    if num_errors > 0:
        batch_size = 18  # 6 righe x 3 colonne

        # Prepara un'immagine "bianca" 224x224 da usare come placeholder
        placeholder_img = Image.new('RGB', (224, 224), color=(255, 255, 255))
        placeholder_array = np.array(placeholder_img)

        for fig_idx, start in enumerate(range(0, num_errors, batch_size), start=1):
            end = start + batch_size
            chunk = sorted_indices[start:end]

            # Creiamo la figura sempre della stessa dimensione
            fig, axes = plt.subplots(6, 3, figsize=(9, 18))
            axes = axes.flatten()

            for i, idx in enumerate(chunk):
                # Carichiamo l'immagine dal generatore
                img_data = test_generator.x_data[idx].astype('uint8')

                # Ridimensioniamo a 224x224
                pil_img = Image.fromarray(img_data).resize((224, 224), Image.BILINEAR)
                resized_img = np.array(pil_img)

                axes[i].imshow(resized_img)
                axes[i].set_title(
                    f"Pred: {class_names_fixed[y_pred[idx]]}\n"
                    f"Conf: {confidences[idx]:.2f}\n"
                    f"True: {class_names_fixed[y_true[idx]]}"
                )
                axes[i].axis("off")

            # Riempie i subplot rimanenti con l'immagine bianca
            for j in range(len(chunk), 18):
                axes[j].imshow(placeholder_array)
                axes[j].axis("off")

            # Se noti comportamenti indesiderati, puoi commentare tight_layout
            plt.tight_layout()

            plt.savefig(f'/content/drive/MyDrive/Colab Notebooks/HPC/finale/{model_name}/finetuning/bias_{fig_idx}.png')
            plt.show()
    else:
        print("Nessun errore con alta confidenza trovato.")

    return probabilities, y_true, y_pred

In [ ]:
keras_models = {
    "ConvNeXt": convnext
}


In [ ]:
keras_models = {
    "ConvNeXt": convnext
}


In [ ]:
import os
import numpy as np
import pandas as pd

def _decode_path(p):
    if isinstance(p, (bytes, np.bytes_)):
        return p.decode('utf-8')
    return str(p)

image_names = [os.path.basename(_decode_path(p)) for p in test_generator_focal_smoot.paths_data]

data = []
for model_name, probabilities, y_true, y_pred in model_results:
    for idx, (true_label, pred_label, prob) in enumerate(zip(y_true, y_pred, probabilities)):
        data.append([model_name, image_names[idx], idx, true_label, pred_label, prob.tolist()])

df = pd.DataFrame(data, columns=["Model", "Image_Path", "Image_Index", "True_Label", "Pred_Label", "Probabilities"])
csv_path = "/content/drive/MyDrive/model_results.csv"
df.to_csv(csv_path, index=False)
print(f"Risultati salvati in: {csv_path}")


# ***PER ABILITARE IL GENERATORE***

# **PER CARICARE IL CSV**

In [ ]:
import os
import numpy as np
import pandas as pd

def _decode_path(p):
    if isinstance(p, (bytes, np.bytes_)):
        return p.decode('utf-8')
    return str(p)

image_names = [os.path.basename(_decode_path(p)) for p in test_generator_focal_smoot.paths_data]

data = []
for model_name, probabilities, y_true, y_pred in model_results:
    for idx, (true_label, pred_label, prob) in enumerate(zip(y_true, y_pred, probabilities)):
        data.append([model_name, image_names[idx], idx, true_label, pred_label, prob.tolist()])

df = pd.DataFrame(data, columns=["Model", "Image_Path", "Image_Index", "True_Label", "Pred_Label", "Probabilities"])
csv_path = "/content/drive/MyDrive/model_results.csv"
df.to_csv(csv_path, index=False)
print(f"Risultati salvati in: {csv_path}")


In [ ]:
print(model_results)

# ***AGREEMENT ADELE***

In [ ]:
import os
import pandas as pd

# Se hai le emozioni in formato TESTUALE nel CSV,
# definisci la tua lista di emozioni. Per esempio:
EMOTIONS_STR = ["ANGRY", "DISGUST", "FEAR", "HAPPY", "NEUTRAL", "SAD", "SURPRISE"]

# Se invece le etichette sono NUMERICHE (0..6) nel CSV,
# puoi definire la tua lista come:
EMOTIONS_NUM = [0, 1, 2, 3, 4, 5, 6]

def Pi_computation(csv_file, image_idx, emotions=EMOTIONS_NUM, label_column="Pred_Label"):
    """
    Calcola:
      - Pi globale di accordo (ritorna come 'overall_pi')
      - Pi per ciascuna emozione (ritorna come dizionario 'pi_dict')

    Parametri
    ---------
    csv_file : str
        Percorso al file CSV che contiene le colonne "Image_Index" e "Pred_Label"
        (oltre eventualmente a "true_label").
    image_idx : str o int
        L'indice dell'immagine di cui vuoi calcolare Pi.
    emotions : list
        Lista delle emozioni possibili (in formati stringa o numerici,
        devono corrispondere a come sono memorizzate in 'Pred_Label').
    label_column : str
        Nome della colonna che contiene la predizione (di default "Pred_Label").

    Ritorna
    -------
    (overall_pi, pi_dict)
      overall_pi : float
          Valore di Pi calcolato sull'immagine specificata (accordo complessivo).
      pi_dict : dict
          Dizionario che mappa ogni emozione al valore di Pi specifico.
    """
    # Leggiamo il CSV
    df = pd.read_csv(csv_file)

    # Filtriamo le righe relative all'indice d'immagine desiderato
    df_image = df[df["Image_Index"] == image_idx]

    # Numero totale di "giudizi" (ossia predizioni disponibili) per questa immagine
    n = len(df_image)


    # Se abbiamo meno di 2 predizioni, la formula c*(c-1)/(n*(n-1)) non è definita
    # (serve almeno n=2 per calcolare l'accordo)
    if n < 2:
        return 0.0, {emo: 0.0 for emo in emotions}

    n_emo = 0
    pi_dict = {}

    # Calcoliamo, per ogni emozione, quante volte è stata predetta
    for emotion in emotions:
        # Conteggio di quante predizioni corrispondono a questa emozione
        c = len(df_image[df_image[label_column] == emotion])

        # Calcolo della quota c*(c-1)/(n*(n-1))
        pi_dict[emotion] = c * (c - 1) / (n * (n - 1))

        # Sommiamo a n_emo la parte c*(c-1) (per calcolare poi Pi globale)
        n_emo += c * (c - 1)

    # Calcolo di Pi complessivo: sum(c*(c-1)) / (n*(n-1))
    overall_pi = n_emo / (n * (n - 1))

    return overall_pi, pi_dict


def Pi_per_emotion(csv_file, image_idx, emotions=EMOTIONS_NUM, label_column="Pred_Label"):
    """
    Calcola solamente il valore di Pi (c*(c-1)/(n*(n-1))) per ogni singola emozione,
    senza ritornare l'accordo complessivo.

    Parametri
    ---------
    csv_file : str
        Percorso al file CSV che contiene le colonne "Image_Index" e "Pred_Label".
    image_idx : str o int
        L'indice dell'immagine di cui vuoi calcolare Pi per emozione.
    emotions : list
        Lista delle emozioni possibili (stringhe o numeri, coerenti con il CSV).
    label_column : str
        Colonna che contiene la predizione.

    Ritorna
    -------
    pi_dict : dict
        Dizionario che mappa ogni emozione al relativo valore di Pi.
    """
    df = pd.read_csv(csv_file)
    df_image = df[df["Image_Index"] == image_idx]

    n = len(df_image)
    pi_dict = {}

    # Se c'è solo una predizione, i valori saranno tutti 0
    if n < 2:
        return {emo: 0.0 for emo in emotions}

    for emotion in emotions:
        c = len(df_image[df_image[label_column] == emotion])
        pi_dict[emotion] = c * (c - 1) / (n * (n - 1))

    return pi_dict


In [ ]:
csv_file="/content/drive/MyDrive/model_results.csv" # PATH DEL CSV
df = pd.read_csv(csv_file)
unique_images = df["Image_Path"].unique()
print(len(unique_images))

In [ ]:
print(df)

In [ ]:
csv_file = "/content/drive/MyDrive/model_results.csv"
# Supponendo che:
# 0 -> ANGRY
# 1 -> DISGUST
# 2 -> FEAR
# 3 -> HAPPY
# 4 -> NEUTRAL
# 5 -> SAD
# 6 -> SURPRISE

# definisci la tua lista di emozioni. Per esempio:
EMOTIONS_STR = ["ANGRY", "DISGUST", "FEAR", "HAPPY", "NEUTRAL", "SAD", "SURPRISE"]

results = []
data = []
df = pd.read_csv(csv_file)
unique_images = df["Image_Index"].unique()

for img_idx in unique_images:


    overall_pi, pi_dict = Pi_computation(
    csv_file,
    img_idx,
    emotions=[0,1,2,3,4,5,6],
    label_column="Pred_Label"
    )
    df_img = df[df["Image_Index"] == img_idx]
    image_path = df.loc[df["Image_Index"] == img_idx, "Image_Path"].values
    # per controllare che ci sia corrispondenza tra indice e path
    '''import matplotlib.pyplot as plt
    import cv2

    # 1️⃣ Ottieni l'indice dell'immagine nel dataset e il path corrispondente
    df_img = df[df["Image_Index"] == img_idx]
    image_path = df_img["Image_Path"].values[0]  # Prende il primo valore trovato

    # 2️⃣ Recupera l'immagine dal test generator (assumendo che test_generator.x_data sia un array NumPy)
    image_array = test_generator_focal_smoot.x_data[img_idx]

    # 3️⃣ Carica l'immagine dal percorso
    image_from_path = cv2.imread(image_path)
    image_from_path = cv2.cvtColor(image_from_path, cv2.COLOR_BGR2RGB)  # OpenCV carica in BGR, quindi converti in RGB

    # 4️⃣ Visualizza entrambe le immagini
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # 🔹 Prima immagine (dal test generator)
    axes[0].imshow(image_array.astype("uint8"))
    axes[0].set_title("Immagine dal Generator")
    axes[0].axis("off")

    # 🔹 Seconda immagine (caricata dal path)
    axes[1].imshow(image_from_path)
    axes[1].set_title("Immagine dal Path")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()'''
    image_path = image_path[0].split('/')[-1]

    # Se per ogni immagine c'è una sola riga, prendi la prima (o unica)
    # occorrenza della colonna true_label:
    true_label = df_img["True_Label"].values[0]
    # Salviamo il risultato in una struttura comoda
    result_row = {
        "Image_Path": image_path,
        "Image_Index": img_idx,
        "Pi_Overall": overall_pi
    }
    # Aggiungiamo le Pi per ogni etichetta (opzionale)
    for lab, val in pi_dict.items():
        result_row[f"Pi_{EMOTIONS_STR[lab]}"] = val


    result_row["True_Label"] = true_label

    results.append(result_row)
    print(f"ANALISI IMMAGINE {image_path}, indice {img_idx}")
    print("Pi complessivo:", overall_pi)
    print("Pi per etichetta:", pi_dict)



In [ ]:
print(len(results))

In [ ]:
# salvare il dataframe in un file csv
df = pd.DataFrame(results)
print(df)
df.to_csv("/content/drive/MyDrive/Colab Notebooks/HPC/finale/agreement_adele/IOA_images.csv", index=False)


In [ ]:
alta_concordanza = len(df[df["Pi_Overall"] > 0.9])
df_ordine = df[df["Pi_Overall"] > 0.9].sort_values(by="Pi_Overall", ascending=False)


label_map = {
    0: "ANGER",
    1: "DISGUST",
    2: "FEAR",
    3: "HAPPINESS",
    4: "NEUTRALITY",
    5: "SADNESS",
    6: "SURPRISE"
}
# Ora costruiamo la lista delle emozioni in formato stringa
emotions = []
for idx, row in df_ordine.iterrows():
    label_num = row["True_Label"]  # Esempio: 0
    label_str = label_map[label_num]  # Esempio: "ANGRY"
    emotions.append(label_str)

# Poi contiamo
count = {}
for emotion in emotions:
    count[emotion] = count.get(emotion, 0) + 1
count = dict(sorted(count.items(), key=lambda item: item[1], reverse=True))
print("Conteggio emozioni (Pi_Overall > 0.9):", count)

# Idem per media_concordanza
media_concordanza = len(df[df["Pi_Overall"] > 0.5])
df_media = df[df["Pi_Overall"] > 0.5]
emotions = []
for idx, row in df_media.iterrows():
    label_num = row["True_Label"]
    label_str = label_map[label_num]
    emotions.append(label_str)

count = {}
for emotion in emotions:
    count[emotion] = count.get(emotion, 0) + 1
count = dict(sorted(count.items(), key=lambda item: item[1], reverse=True))
print("Conteggio emozioni (Pi_Overall > 0.5):", count)

# E idem per bassa_concordanza (<= 0.3)
bassa_concordanza = len(df[df["Pi_Overall"] <= 0.3])
df_bassa = df[df["Pi_Overall"] <= 0.3]
emotions = []
for idx, row in df_bassa.iterrows():
    label_num = row["True_Label"]
    label_str = label_map[label_num]
    emotions.append(label_str)

count = {}
for emotion in emotions:
    count[emotion] = count.get(emotion, 0) + 1
count = dict(sorted(count.items(), key=lambda item: item[1], reverse=True))
print("Conteggio emozioni (Pi_Overall <= 0.3):", count)

print(f"Alta concordanza: {alta_concordanza} su {len(df)}, {alta_concordanza/len(df)*100:.2f}%")
print(f"Media concordanza: {media_concordanza} su {len(df)}, {media_concordanza/len(df)*100:.2f}%")
print(f"Bassa concordanza: {bassa_concordanza} su {len(df)}, {bassa_concordanza/len(df)*100:.2f}%")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Filtra il DataFrame e ordina
maggiori = df[df["Pi_Overall"] > 0]       # Pi > 0
maggiori = maggiori.sort_values(by="Pi_Overall", ascending=False)

print("Numero di righe con Pi_Overall > 0:", len(maggiori))
print("Percentuale rispetto al totale:", len(maggiori)/len(df)*100, "%")

count = {}
images_names = []
num = 0

for idx, row in maggiori.iterrows():
    # Ora recuperi l'emozione REALE dalla colonna True_Label,
    # invece di fare split("_")[2] sul nome file.
    #emotion_real = row["True_Label"]  # Se è "ANGRY", "HAPPY", ecc.

    # Se la True_Label fosse numerica (0..6),
    # usi un mapping:
    label_map = {
    0: "ANGRY",
    1: "DISGUST",
    2: "FEAR",
    3: "HAPPY",
    4: "NEUTRAL",
    5: "SAD",
    6: "SURPRISE"
    }
    emotion_real = label_map[row["True_Label"]]

    # Trova il valore di Pi massimo tra Pi_Angry ... Pi_Surprise
    pi_max = row[[f"Pi_{label_map[0]}",
                  f"Pi_{label_map[1]}",
                  f"Pi_{label_map[2]}",
                  f"Pi_{label_map[3]}",
                  f"Pi_{label_map[4]}",
                  f"Pi_{label_map[5]}",
                  f"Pi_{label_map[6]}"]].max()

    # Scopri quale colonna ha quel massimo
    emotion_col = row[[f"Pi_{label_map[0]}",
                      f"Pi_{label_map[1]}",
                      f"Pi_{label_map[2]}",
                      f"Pi_{label_map[3]}",
                      f"Pi_{label_map[4]}",
                      f"Pi_{label_map[5]}",
                      f"Pi_{label_map[6]}"]].idxmax()
    # Esempio: emotion_col = "Pi_Angry"

    # Se vuoi il nome dell'emozione predetta, puoi togliere "Pi_" e mettere in maiuscolo
    # Pi_Angry -> ANGRY
    emotion_pred = emotion_col.replace("Pi_", "").upper()

    # Se la real è minuscola o maiuscola mescolata, uniforma:
    emotion_real_up = emotion_real.upper()

    # Controlla se l'emozione di massimo accordo è diversa da quella reale
    if emotion_pred != emotion_real_up:
        # Inizializza la sotto-dict se non esiste
        count.setdefault(emotion_real_up, {})
        # Incrementa il contatore per (emozione_reale -> emozione_predetta)
        count[emotion_real_up][emotion_pred] = count[emotion_real_up].get(emotion_pred, 0) + 1
        num += 1

        # Salvi (per esempio) il nome del file, se presente in colonna "Image_Name"
        # Se la colonna "Image_Name" non esiste più, puoi salvare "Image_Index"
        # o quello che hai.
        images_names.append(row["Image_Index"])

        # Se vuoi mostrare l'immagine, e se la directory ha struttura
        #   bosphorus_test_HQ/EMOZIONE_REALE/image_name
        # allora:
        #   img_path = os.path.join(image_path, emotion_real_up, row["Image_Name"])
        #   img = plt.imread(img_path)
        #   plt.imshow(img)
        #   plt.axis("off")
        #   plt.show()

print("Conteggio (emozione reale -> emozione predetta errata):")
print(count)
print("Numero totale di disallineamenti:", num)
print("Lista di immagini su cui c'è disallineamento:", images_names)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

csv_file = "/content/drive/MyDrive/model_results.csv"
df_count = pd.read_csv(csv_file)
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/HPC/finale/agreement_adele/IOA_images.csv")
# Filtra il DataFrame e ordina
maggiori = df[df["Pi_Overall"] > 0]       # Pi > 0
maggiori = maggiori.sort_values(by="Pi_Overall", ascending=False)

print("Numero di righe con Pi_Overall > 0:", len(maggiori))
print("Percentuale rispetto al totale:", len(maggiori)/len(df)*100, "%")

EMOTIONS = ["ANGRY","DISGUST","FEAR","HAPPY","NEUTRAL","SAD","SURPRISE"]
EMOTIONS_NUM = [0,1,2,3,4,5,6]

label_map = {
    0: "ANGER",
    1: "DISGUST",
    2: "FEAR",
    3: "HAPPINESS",
    4: "NEUTRALITY",
    5: "SADNESS",
    6: "SURPRISE"
}

emotion_map = {
    "ANGER": "ANGRY",
    "DISGUST": "DISGUST",
    "FEAR": "FEAR",
    "HAPPINESS": "HAPPY",
    "NEUTRALITY": "NEUTRAL",
    "SADNESS": "SAD",
    "SURPRISE": "SURPRISE"
}

def calc_total_wrong(row):
    """Data una riga di 'maggiori', trova quante predizioni (modelli)
    hanno sbagliato in df_count per quell'immagine."""
    image_idx = row["Image_Index"]
    df_img_all = df_count[df_count["Image_Index"] == image_idx]
    wrong_mask = df_img_all["Pred_Label"] != df_img_all["True_Label"]
    df_wrong = df_img_all[wrong_mask]
    return len(df_wrong)

def has_wrong_max(row):
    """Ritorna True se l'emozione di Pi massimo NON è quella reale. Altrimenti False."""
    real_emo_str = label_map[row["True_Label"]]  # Es. 3 -> "HAPPINESS"
    all_pi_cols = ["Pi_ANGRY","Pi_DISGUST","Pi_FEAR","Pi_HAPPY","Pi_NEUTRAL","Pi_SAD","Pi_SURPRISE"]
    all_row_vals = [row[c] for c in all_pi_cols]
    max_idx = np.argmax(all_row_vals)
    emo_max_str = all_pi_cols[max_idx].replace("Pi_","").upper()  # Es. "ANGRY","HAPPY", ...
    return (emo_max_str != real_emo_str)

# Aggiungiamo la colonna "Total_Wrong"
maggiori["Total_Wrong"] = maggiori.apply(calc_total_wrong, axis=1)

# Filtriamo per mostrare solo le immagini con almeno 1 errore
maggiori_wrong = maggiori[maggiori["Total_Wrong"] > 0]
num_images = len(maggiori_wrong)
print("Numero immagini con errori:", num_images)

# Filtriamo in base a 'has_wrong_max'
maggiori_wrong_filter = maggiori_wrong[maggiori_wrong.apply(has_wrong_max, axis=1)]
print("Numero immagini dove la massima Pi != reale:", len(maggiori_wrong_filter))

# Ordiniamo per Pi_Overall discendente
df_to_plot = maggiori_wrong_filter.sort_values(by="Pi_Overall", ascending=False).copy()
df_to_plot.reset_index(drop=True, inplace=True)

num_images = len(df_to_plot)
print(f"Totale immagini da plottare (massima Pi != reale): {num_images}")

# Creiamo una lista di dizionari con i dati necessari per la visualizzazione
plot_data = []
for i, row in df_to_plot.iterrows():
    image_idx = row["Image_Index"]
    image_path = row["Image_Path"]  # path assoluto o relativo
    true_label_val = row["True_Label"]
    emotion_real = label_map[true_label_val]

    pi_total = row["Pi_Overall"]
    total_wrong = row["Total_Wrong"]
    if total_wrong <=1:
      continue

    # Troviamo l'emozione col Pi massimo su TUTTE le colonne
    all_pi_cols = ["Pi_ANGRY","Pi_DISGUST","Pi_FEAR","Pi_HAPPY","Pi_NEUTRAL","Pi_SAD","Pi_SURPRISE"]
    all_row_vals = [row[c] for c in all_pi_cols]
    max_idx = np.argmax(all_row_vals)
    emotion_max = label_map[max_idx]  # Esempio: 0->"ANGER",1->"DISGUST", etc.
    pi_max = all_row_vals[max_idx]

    df_img_all = df_count[df_count["Image_Index"] == image_idx]
    wrong_mask = df_img_all["Pred_Label"] != df_img_all["True_Label"]
    df_wrong = df_img_all[wrong_mask]

    if len(df_wrong) > 0:
        most_frequent = df_wrong["Pred_Label"].value_counts().idxmax()
    else:
        most_frequent = None

    details_str = "Misclassifications (wrong only):\n"
    for em_num in EMOTIONS_NUM:
        count_em = sum(df_wrong["Pred_Label"] == em_num)
        details_str += f" - {label_map[em_num]}: {count_em}\n"

    plot_data.append({
        "image_idx": image_idx,
        "image_path": image_path,
        "emotion_real": emotion_real,
        "pi_total": pi_total,
        "total_wrong": total_wrong,
        "emotion_max": emotion_max,
        "pi_max": pi_max,
        "details_str": details_str,
        "most_frequent": most_frequent
    })

# Parametri per la visualizzazione
IMAGES_PER_FIG = 18  # 6x3
num_chunks = math.ceil(len(plot_data) / IMAGES_PER_FIG)
img_num = 0

# Creiamo un placeholder bianco 224x224 in caso l'ultimo batch abbia meno di 18 immagini
placeholder_img = Image.new('RGB', (224, 224), color=(255, 255, 255))
placeholder_array = np.array(placeholder_img)

for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * IMAGES_PER_FIG
    end_idx = start_idx + IMAGES_PER_FIG
    chunk = plot_data[start_idx:end_idx]

    if not chunk:
        break

    # Creiamo figura 6x3 = 18 subplot, dimensioni (15, 30) per esempio
    fig, axes = plt.subplots(6, 3, figsize=(12, 30))
    axes_flat = axes.flatten()

    for i, data_item in enumerate(chunk):
        ax = axes_flat[i]

        image_path = data_item["image_path"]
        emotion_real = data_item["emotion_real"]
        emotion_max = data_item["emotion_max"]
        pi_max = data_item["pi_max"]
        total_wrong = data_item["total_wrong"]
        pi_total = data_item["pi_total"]
        details_str = data_item["details_str"]
        most_frequent = data_item["most_frequent"]
        if most_frequent is not None:
            most_frequent_str = EMOTIONS[most_frequent]
        else:
            most_frequent_str = 'None'

        # Carichiamo l'immagine dal percorso ricalcolato (se necessario)
        path_correct = os.path.join(
            '/content/drive/MyDrive/Colab Notebooks/datasets/dataset_giusto/test',
            emotion_map[emotion_real],
            image_path
        )
        img = Image.open(path_correct)

        # Ridimensioniamo l'immagine a 224x224 per coerenza
        img_resized = img.resize((224, 224), Image.BILINEAR)
        img_array = np.array(img_resized)

        ax.imshow(img_array)
        ax.axis('off')

        ax.set_title(
            f"Misclassified by {int(total_wrong)} annotators\n"
            f"Overall Agreement: {pi_total*100:.2f}%\n"
            f"Agreement w/ {emotion_max}: {pi_max*100:.2f}%\n"
            f"Most common misclassification: {most_frequent_str}\n"
            f"Correct class: {emotion_real}\n\n"
            f"{details_str}",
            fontsize=10
        )

    # Se il chunk ha meno di 18 immagini, riempi i subplot restanti con placeholder
    for j in range(len(chunk), IMAGES_PER_FIG):
        axes_flat[j].imshow(placeholder_array)
        axes_flat[j].axis('off')

    plt.subplots_adjust(wspace=0.05,hspace = 1.5)
    img_num += 1
    plt.savefig(f'/content/drive/MyDrive/Colab Notebooks/HPC/finale/agreement_adele/agreement_{img_num}.png')
    plt.show()

print("DONE. Tutti i plot 6x3 (18 immagini) salvati correttamente.")

# ***AGREEMENT FEDE***

In [ ]:
from collections import defaultdict

error_counts = defaultdict(int)

for _, _, y_true, y_pred in model_results:
    errors = np.where(y_true != y_pred)[0]  # Indici delle immagini sbagliate
    for idx in errors:
        error_counts[idx] += 1  # Conta quanti modelli sbagliano questa immagine

# 🔥 Stampa le immagini più difficili
sorted_errors = sorted(error_counts.items(), key=lambda x: x[1], reverse=True)
print("\n Immagini più difficili (sbagliate dal maggior numero di modelli):")
for idx, count in sorted_errors[:10]:  # Mostra le prime 10
    print(f"Immagine {idx} sbagliata da {count} modelli")

error_predictions = defaultdict(set)

for model_name, _, y_true, y_pred in model_results:
    errors = np.where(y_true != y_pred)[0]  # Indici delle immagini sbagliate
    for idx in errors:
        error_predictions[idx].add(y_pred[idx])  # Registra le classi predette errate

# Stampa le immagini in cui i modelli hanno fatto lo stesso errore
print("\n Errori comuni tra i modelli:")
for idx, predicted_classes in error_predictions.items():
    if len(predicted_classes) > 1:
        print(f"Immagine {idx}: modelli hanno predetto {predicted_classes}")

In [ ]:
import numpy as np

for model_name, probabilities, y_true, y_pred in model_results:
    print(f"\n Analisi delle incertezze per il modello: {model_name}\n")

    for idx, (true_label, pred_label, prob) in enumerate(zip(y_true, y_pred, probabilities)):
        if pred_label != true_label:  # Consideriamo solo le immagini errate
            top_2_indices = np.argsort(prob)[-2:][::-1]  # Ordiniamo le due classi con probabilità più alta
            top_2_classes = [class_names[i] for i in top_2_indices]  # Convertiamo in nomi di classi
            top_2_probs = prob[top_2_indices]  # Otteniamo le probabilità corrispondenti

            print(f"   Immagine {idx}:")
            print(f"   Classe reale: {class_names[true_label]}")
            print(f"   Classe predetta: {top_2_classes[0]} (Conf: {top_2_probs[0]:.2f})")
            print(f"   Seconda scelta: {top_2_classes[1]} (Conf: {top_2_probs[1]:.2f})")

            # Se il modello è molto incerto (differenza < 0.1), lo segnaliamo
            if abs(top_2_probs[0] - top_2_probs[1]) < 0.1:
                print(f"  Il modello è in forte dubbio tra {top_2_classes[0]} e {top_2_classes[1]}")

# ***per vedere l'incertezza dei modelli tra prima e seconda classe più probabile***

In [ ]:
print(model_results[0])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image  # Per ridimensionare le immagini e creare placeholder

threshold = 0.1  # Differenza minima tra la prima e la seconda probabilità
batch_size = 18  # 6 righe x 3 colonne = 18 immagini per figura
class_names_fixed = ['ANGER', 'DISGUST', 'FEAR', 'HAPPINESS', 'NEUTRALITY', 'SADNESS', 'SURPRISE']
# Creiamo un'immagine placeholder 224x224 (bianca)
placeholder_img = Image.new('RGB', (224, 224), color=(255, 255, 255))
placeholder_array = np.array(placeholder_img)

for model_name, probabilities, y_true, y_pred in model_results:
    print(f"\nAnalisi delle incertezze per il modello: {model_name}\n")

    uncertain_images = []

    # Scorriamo tutte le immagini
    for idx, (true_label, pred_label, prob) in enumerate(zip(y_true, y_pred, probabilities)):
        # Consideriamo solo i casi in cui il modello sbaglia
        if pred_label != true_label:
            # Cerchiamo le due classi più probabili
            top_2_indices = np.argsort(prob)[-2:][::-1]
            top_2_classes = [class_names_fixed[i] for i in top_2_indices]
            top_2_probs = prob[top_2_indices]

            # Se la differenza tra prima e seconda probabilità < threshold => immagine "incerta"
            if abs(top_2_probs[0] - top_2_probs[1]) < threshold:
                print(f"Immagine {idx} è molto incerta tra {top_2_classes[0]} e {top_2_classes[1]}")
                print(f"Classe predetta: {top_2_classes[0]} (Conf: {top_2_probs[0]:.2f})")
                print(f"Seconda scelta: {top_2_classes[1]} (Conf: {top_2_probs[1]:.2f})")
                uncertain_images.append((idx, top_2_classes, top_2_probs, true_label))

    # Se abbiamo immagini incerte, mostriamole in figure 6x3
    if uncertain_images:
        num_images = len(uncertain_images)
        print(f"\nNumero totale di immagini incerte per il modello {model_name}: {num_images}\n")

        # Suddividiamo le immagini in blocchi di 18
        for fig_idx, start in enumerate(range(0, num_images, batch_size), start=1):
            end = start + batch_size
            chunk = uncertain_images[start:end]

            # Creiamo una figura di dimensioni fisse (9x18 pollici)
            fig, axes = plt.subplots(6, 3, figsize=(9, 18))
            axes = axes.flatten()

            for i, (img_idx, top_classes, top_probs, true_label) in enumerate(chunk):
                # Carichiamo l'immagine (shape: H,W,C)
                img_data = test_generator_focal_smoot.x_data[img_idx].astype('uint8')

                # Ridimensioniamo fisicamente l’immagine a 224x224
                pil_img = Image.fromarray(img_data).resize((224, 224), Image.BILINEAR)
                resized_img = np.array(pil_img)

                # Mostriamo l'immagine ridimensionata
                axes[i].imshow(resized_img)
                axes[i].axis('off')
                axes[i].set_title(
                    f"Pred: {top_classes[0]} ({top_probs[0]:.2f})\n"
                    f"Sec: {top_classes[1]} ({top_probs[1]:.2f})\n"
                    f"True: {class_names_fixed[true_label]}"
                )

            # Riempie i subplot non usati con il placeholder bianco
            for j in range(len(chunk), batch_size):
                axes[j].imshow(placeholder_array)
                axes[j].axis('off')

            plt.tight_layout()
            plt.savefig(f'/content/drive/MyDrive/Colab Notebooks/HPC/finale/{model_name}/uncertain_{fig_idx}.png')
            plt.show()
    else:
        print(f"Nessuna immagine incerta da visualizzare per il modello {model_name}.\n")

# ***per vedere se i modelli sono d'accordo tra di loro nel predirre la classe errata***

In [ ]:
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

class_names = ['ANGRY', 'DISGUST', 'FEAR', 'HAPPY', 'NEUTRAL', 'SAD', 'SURPRISE']
# Raccogliamo i dettagli dei modelli che sbagliano
wrong_models = defaultdict(list)  # key = image idx, value = list of (model_name, predicted_class)

for (model_name, probabilities, y_true, y_pred) in model_results:
    for idx, (true_label, pred_label) in enumerate(zip(y_true, y_pred)):
        if pred_label != true_label:
            wrong_models[idx].append((model_name, pred_label))

# Calcoliamo l'accordo di misclassificazione
image_agreements = []
for idx, wrong_list in wrong_models.items():
    total_wrong = len(wrong_list)
    wrong_preds = [pred_label for (_, pred_label) in wrong_list]
    count_preds = Counter(wrong_preds)
    most_common_class, freq_most_common = count_preds.most_common(1)[0]
    agreement = freq_most_common / total_wrong
    image_agreements.append((idx, agreement, most_common_class, total_wrong))

# Ordiniamo per numero di modelli che sbagliano
image_agreements.sort(key=lambda x: x[3], reverse=True)

# Parametri per la griglia
IMAGES_PER_FIG = 15  # 5 x 3
cols = 3
rows = 5

num_images = len(image_agreements)
if num_images > 0:
    # Cicliamo a gruppi di 25
    for start_idx in range(0, num_images, IMAGES_PER_FIG):
        chunk = image_agreements[start_idx:start_idx+IMAGES_PER_FIG]

        # Creiamo una figura per questo blocco di immagini
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
        axes = axes.flatten()

        for i, ax in enumerate(axes):
            if i < len(chunk):
                idx, agreement, most_common_class, total_wrong = chunk[i]

                wrong_list = wrong_models[idx]  # lista di (model_name, predicted_class)
                details_str = "Misclassifications:\n"
                for (m_name, pred_class) in wrong_list:
                    details_str += f" - {m_name}: {class_names[pred_class]}\n"

                img = test_generator_focal_smoot.x_data[idx]
                true_label_idx = np.argmax(test_generator_focal_smoot.y_data[idx])

                ax.imshow(img.astype('uint8'))
                ax.axis('off')
                ax.set_title(
                    f"Misclassified by {total_wrong} models\n"
                    f"Agreement: {agreement*100:.2f}%\n"
                    f"Most common misclassification: {class_names[most_common_class]}\n"
                    f"Correct class: {class_names[true_label_idx]}\n\n"
                    f"{details_str}"
                )
            else:
                ax.axis('off')

        plt.tight_layout()
        #plt.savefig(f"/content/drive/MyDrive/Colab Notebooks/HPC/finale/agreement_page_{start_idx}.png") # per salvare l'immagine
        plt.show()

else:
    print("No misclassified images to display.")